In [247]:
import pickle
import pandas as pd
import numpy as np
import pickle


In [248]:
with open("../data/01-result/drugs_df.pkl","rb") as f:
    drugs_df =pickle.load(f)
with open("../data/01-result/indications_df.pkl","rb") as f:
    indications_df = pickle.load(f)
with open("../data/01-result/trials_df.pkl","rb") as f:
    trials_df =pickle.load(f)
with open("../data/01-result/diseases_df.pkl","rb") as f:
    diseases_df = pickle.load(f)

# ABSOLUTE TODO: Engineer labels

# Curate trials and indications

In [249]:
# Extract interesting data from clinical trials - dates and results
final_trials_df = trials_df.drop(columns=["annotationSection", "documentSection"]) # type: ignore
final_trials_df["nct_id"] = None 
final_trials_df["status"] = None
final_trials_df["phase"] = None
final_trials_df["success"] = None
final_trials_df["median_p_value"] = None
final_trials_df["p_value_list"] = None
for i, study in final_trials_df.iterrows():
    # TODO: choose better dates
    final_trials_df.loc[i, "nct_id"] = study["protocolSection"]["identificationModule"]["nctId"]
    final_trials_df.loc[i, "status"] = study["protocolSection"]["statusModule"]["overallStatus"]
    final_trials_df.loc[i, "start_date"] = study["protocolSection"]["statusModule"].get("startDateStruct",{"date":None})["date"]
    final_trials_df.loc[i, "end_date"] = study["protocolSection"]["statusModule"].get("completionDateStruct",{"date":None})["date"]
    final_trials_df.loc[i, "why_stopped"] = study["protocolSection"]["statusModule"].get("whyStopped", None)
    if "designModule" in study["protocolSection"]:
        phases = study["protocolSection"]["designModule"].get("phases",[])
        phases = [int(phase[-1]) for phase in phases if phase != "NA"]
        if phases:
            final_trials_df.loc[i, "phase"] = np.max(phases)
    # Get p-values
    # if "conditionBrowseModule" in study["derivedSection"]:
    #     found_mesh = [mesh['id'] for mesh in  study["derivedSection"]["conditionBrowseModule"]["meshes"]]
    #     if True in [mesh in mesh_ids for mesh in found_mesh]:
    if study["hasResults"]:
        measures = study["resultsSection"]["outcomeMeasuresModule"]["outcomeMeasures"]
        p_values = [measure["analyses"][0]["pValue"] for measure in measures if "analyses" in measure if "pValue" in measure["analyses"][0]]
        p_values = [float(''.join([ch for ch in p if ch.isdigit() or ch=="."])) for p in p_values]
        if len(p_values) > 0:
            final_trials_df.at[i, "p_value_list"] = p_values
final_trials_df = final_trials_df.drop(columns=["derivedSection","protocolSection"])
first_columns = ['nct_id','success',"median_p_value",'phase','status', 'hasResults','why_stopped']
final_trials_df = final_trials_df[first_columns + [c for c in final_trials_df.columns if c not in first_columns]]

# TODO: criteria for fail and sucess!
for i, study in final_trials_df.iterrows():
    if study["p_value_list"]:
        medp = np.median(study["p_value_list"])
        prop05 = (np.array(study["p_value_list"]) < 0.05).sum() / len(study["p_value_list"])
        conflicting = (np.min(study["p_value_list"]) < 0.05) and ((np.array(study["p_value_list"]) > 0.2).sum() / len(study["p_value_list"]) > 0.5)
        if medp <= 0.05 or (medp <= 0.1 and prop05 >= 0.5):
            final_trials_df.loc[i, "success"] = "success"
        # else: 
        #     final_trials_df.loc[i, "success"] = "fail"
        elif 0.05 < medp <= 0.20 or (0.10 < medp <= 0.50 and 0.1 <= prop05 < 0.5) or conflicting:
            final_trials_df.loc[i, "success"] = "unknown"
        elif medp > 0.2 and prop05 < 0.1:
            final_trials_df.loc[i, "success"] = "fail"

In [250]:
# Add this trial info to corresponding indications
final_indications_df = indications_df.drop(columns=["drugind_id", "mesh_heading", "parent_molecule_chembl_id"])
final_indications_df["nct_evidence"] = None

for i, indication in final_indications_df.iterrows():
    # TODO: finish logic with success
    # add dates 
    if indication["nct_ids"]:
        success_list = []
        for nct_id in indication["nct_ids"]:
            success = final_trials_df.loc[final_trials_df["nct_id"]==nct_id,"success"].values
            if len(success)>0:
                if success[0] is not None:
                    success_list.append(success[0] + str(final_trials_df.loc[final_trials_df["nct_id"]==nct_id,"phase"].values[0]))
        final_indications_df.at[i, "nct_evidence"] = success_list
final_indications_df.rename(columns={"molecule_chembl_id":"drug_id"}, inplace=True)
first_columns = ['drug_id','efo_term','efo_id','mesh_id', 'max_phase_for_ind','nct_evidence']
final_indications_df= final_indications_df[first_columns + [c for c in final_indications_df.columns if c not in first_columns]]
final_indications_df["disease_id"] = final_indications_df["efo_id"]

# Set up the label

In [251]:
for i, row in final_indications_df.iterrows():
    if final_indications_df.loc[i,"max_phase_for_ind"] == "4.0":
        final_indications_df.at[i, "overall_success"] = True
    elif final_indications_df.loc[i,"nct_evidence"]:
        all_results = final_indications_df.loc[i,"nct_evidence"]
        highest_phase = np.max([int(result[-1]) for result in all_results if "None" not in result])
        latest_results = [result[:-1] for result in all_results if "None" not in result and int(result[-1]) == highest_phase]
        
        # TODO: decide on how do you assign fail
        if "success" in latest_results and not "fail" in latest_results:
            final_indications_df.at[i, "overall_success"] = True
        elif "fail" in latest_results:
            final_indications_df.at[i, "overall_success"] = False
        else:
            final_indications_df.at[i, "overall_success"] =  None # False
    else:
        final_indications_df.at[i, "overall_success"] =  None # False
    
    # TODO: 
    if final_indications_df.loc[i,"max_phase_for_ind"] not in ["2.0","3.0","4.0"]:
        final_indications_df.at[i, "overall_success"] = False
final_indications_df["overall_success"] = final_indications_df["overall_success"].astype("boolean")

first_columns = ["drug_id", "disease_id", "overall_success", "nct_evidence", "max_phase_for_ind"] 
final_indications_df= final_indications_df[first_columns + [c for c in final_indications_df.columns if c not in first_columns]]

# Feature engineering based on pathways

In [252]:
# # TODO: generate other features
# for i, row in merged_df.iterrows():
#     merged_df.at[i, "n_shared_pathways"] = len(set(merged_df.loc[i,"drug_pathways"]) and set(merged_df.loc[i,"disease_pathways"]))

# convert lists of IDs to space separated strings for TF-IDF
drugs_df["drug_path_str"] = drugs_df["drug_pathways"].apply(lambda x: " ".join(x))
diseases_df["disease_path_str"] = diseases_df["disease_pathways"].apply(lambda x: " ".join(x))
corpus = pd.concat([drugs_df['drug_path_str'], diseases_df['disease_path_str']], ignore_index=True)

from sklearn.feature_extraction.text import TfidfVectorizer
# TF-IDF helps downweighting housekeeping pathways and highlights informative ones
vectorizer = TfidfVectorizer(token_pattern=r"[A-Za-z0-9_-]+")
tfidf_matrix = vectorizer.fit_transform(corpus)

from sklearn.decomposition import TruncatedSVD
# TODO: decide on nb of components
svd = TruncatedSVD(n_components=50, random_state=42)
latent = svd.fit_transform(tfidf_matrix)

n_drugs = len(drugs_df)
drug_features = latent[:n_drugs]
disease_features = latent[n_drugs:]

for i in range(drug_features.shape[1]):
    drugs_df[f"drug_path_{i}"] = drug_features[:,i]

for i in range(disease_features.shape[1]):
    diseases_df[f"disease_path_{i}"] = disease_features[:,i]

# Save results

In [253]:
with open("../data/02-result/drugs_df.pkl","wb") as f:
    pickle.dump(drugs_df,f)
with open("../data/02-result/diseases_df.pkl","wb") as f:
    pickle.dump(diseases_df,f)
with open("../data/02-result/trials_df.pkl","wb") as f:
    pickle.dump(final_trials_df,f)
with open("../data/02-result/indications_df.pkl","wb") as f:
    pickle.dump(final_indications_df,f)
